In [ ]:
import datetime
import os
import pickle

import mlflow
import numpy as np
from joblib import dump
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# rcv1 = fetch_rcv1()

In [ ]:

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

DATA_PATH = os.path.join(DATA_DIR, "data.pickle")
TARGET_PATH = os.path.join(DATA_DIR, "target.pickle")

if os.path.exists(DATA_PATH) and os.path.exists(TARGET_PATH):
    print("[INFO] Loading X, y from existing pickle files...")
    with open(DATA_PATH, "rb") as f_data:
        X = pickle.load(f_data)
    with open(TARGET_PATH, "rb") as f_target:
        y = pickle.load(f_target)
else:
    print("[INFO] Pickle files not found. Loading Breast Cancer dataset and saving...")
    data = load_breast_cancer()
    X = data.data
    y = data.target

    with open(DATA_PATH, "wb") as f_data:
        pickle.dump(X, f_data)
    with open(TARGET_PATH, "wb") as f_target:
        pickle.dump(y, f_target)

print(f"[INFO] Data shape: X={X.shape}, y={y.shape}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
) 

In [ ]:

mlflow.set_tracking_uri("./mlruns")
dataset_name = "BreastCancerWisconsin"
current_time = datetime.datetime.now().strftime("%y%m%d_%H%M%S")
experiment_name = f"{dataset_name}_TEST_{current_time}"
experiment_id = mlflow.create_experiment(experiment_name)

with mlflow.start_run(
    experiment_id=experiment_id,
    run_name=f"{dataset_name}_test_run",
):
    params = {
        "dataset_name": dataset_name,
        "n_samples_total": X.shape[0],
        "n_features": X.shape[1],
        "n_samples_train": X_train.shape[0],
        "n_samples_test": X_test.shape[0],
    }
    mlflow.log_params(params)

    # ----------------- 4) Build and train model (same style as train_model.py) -----------------

    pipeline = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("clf", RandomForestClassifier(random_state=0)),
        ]
    )

    pipeline.fit(X_train, y_train)
    print("[INFO] Test script: model training complete.")

    # ----------------- 5) Evaluate -----------------

    # Default threshold 0.5 on predicted probabilities
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    y_pred_default = (y_proba >= 0.5).astype(int)

    y_pred_train = pipeline.predict(X_train)

    train_accuracy = accuracy_score(y_train, y_pred_train)
    train_f1 = f1_score(y_train, y_pred_train)

    test_accuracy_default = accuracy_score(y_test, y_pred_default)
    test_f1_default = f1_score(y_test, y_pred_default)

    # Simple threshold search for best F1 (same idea as train_model.py)
    best_f1 = -1.0
    best_threshold = 0.5

    for thr in np.linspace(0.1, 0.9, 9):
        y_pred_thr = (y_proba >= thr).astype(int)
        f1_thr = f1_score(y_test, y_pred_thr)
        if f1_thr > best_f1:
            best_f1 = f1_thr
            best_threshold = thr

    mlflow.log_metrics(
        {
            "train_accuracy": train_accuracy,
            "train_f1": train_f1,
            "test_accuracy_default": test_accuracy_default,
            "test_f1_default": test_f1_default,
            "test_f1_best": best_f1,
        }
    )
    mlflow.log_param("best_threshold", float(best_threshold))

    print(
        f"[INFO] Test script metrics -> "
        f"train_acc={train_accuracy:.4f}, train_f1={train_f1:.4f}, "
        f"test_acc_default={test_accuracy_default:.4f}, "
        f"test_f1_default={test_f1_default:.4f}, "
        f"best_f1={best_f1:.4f} @ thr={best_threshold:.2f}"
    )

    # ----------------- 6) Save model -----------------

    MODELS_DIR = "models"
    os.makedirs(MODELS_DIR, exist_ok=True)

    model_filename = f"test_model_{current_time}_rf_pipeline.joblib"
    model_path = os.path.join(MODELS_DIR, model_filename)

    dump(pipeline, model_path)
    print(f"[INFO] Test script saved model to: {model_path}")             